# The Architecture Pareto Frontier

*What "which one should we use?" actually depends on.*

The predecessor measured six named RAG architectures and found no row that wins both regimes. It
reported cost in **two currencies** — similarity operations and generator calls — and refused to
collapse them, because the exchange rate between them is exactly what a cost model should not
invent. That left a Pareto set rather than a winner, and this notebook is what you do with it.

The answer to *which architecture should we deploy* turns out to require three more things, none
of which is a property of any architecture: the **workload mixture**, the **price ratio** between
the two currencies, and the **amortization horizon**. This walks through what each one does.

Nothing is re-measured here. The module imports the predecessor's arms and runs them once into a
6 × 6 × 3 table; everything after that is arithmetic.

```
uv run --with numpy --with scipy --with jupyter \
    jupyter execute notebooks/rag-architecture-pareto/01_rag_architecture_pareto.ipynb
```


In [ ]:
import pathlib
import sys

import numpy as np

sys.path.insert(0, str(pathlib.Path.cwd() if (pathlib.Path.cwd() / "rag_architecture_pareto.py").exists()
                       else pathlib.Path.cwd() / "notebooks" / "rag-architecture-pareto"))
import rag_architecture_pareto as P

print(f"arms       : {', '.join(P.ARM_NAMES)}")
print(f"conditions : {', '.join(P.CONDITIONS)}")
t = P.measure()
print(f"the table  : {t['Q'].shape[0]} x {t['Q'].shape[1]} x 3 = "
      f"{t['Q'].size * 3} numbers, and everything below is arithmetic on them")

## 1. Cost is a pair, and it is not a constant

An arm's cost on a workload lands in $\mathbb{R}^2$, which has no total order — "cheaper" stops
being a comparison two architectures can always settle.

Worse for the familiar comparison table, the operation count genuinely depends on the *kind* of
question for an adaptive arm, because the thing that makes it adaptive is a gate that fires on
some questions and not others.

In [ ]:
print(P._fmt_table())
print()
print("spread across question kinds (within the local regime, so this is adaptivity not corpus size):")
for a in P.ARM_NAMES:
    flag = "  <- no single cost figure is right for this arm" if P.cost_spread(a) >= 5 else ""
    print(f"  {a:11s} {P.cost_spread(a):6.2f}x{flag}")

## 2. Domination, and what it is a property of

An arm dominates another when it is at least as good on every axis and strictly better on one.
Both sides of every one of those inequalities is a linear functional of the workload — so the
relation carries the workload with it.

On the mixture the predecessor implicitly reported at, one arm is beaten on *every axis at once*.

In [ ]:
w = P.local_only()
print("Pareto set on the aggregate local workload:", P.pareto_set(w))
print()
beaten = P.dominated_by("agentic", w)
q, o, c = P.profile(w)
ai, ni = P.ARM_NAMES.index("agentic"), P.ARM_NAMES.index("naive")
print(f"agentic is dominated by {beaten}")
print(f"  quality {q[ai]:.3f} < {q[ni]:.3f}   ops {o[ai]:.0f} > {o[ni]:.0f}   calls {c[ai]:.2f} > {c[ni]:.2f}")
print("  -> beaten on every axis. By the Pareto criterion, never deploy it.")

And then the mixture moves.

In [ ]:
pure = P.one_hot("bridge")
print("on a workload of nothing but bridge questions:")
print(f"  Pareto set   : {P.pareto_set(pure)}")
print(f"  would deploy : {P.winner(pure, lam=0.0)}")
print()
print(f"the crossover: agentic becomes the arm to deploy once the bridge share reaches "
      f"{P.bridge_crossover():.2f}")
print()
print("The aggregate numbers were not wrong. They were computed correctly, on a mixture,")
print("and they support a conclusion that a different mixture reverses.")

## 3. The winner map partitions the simplex

Fix a price for generation and a price for cost in units of quality, and you have the selection
rule every cost-aware evaluation computes. The map from workload to winner partitions the
simplex, and what that partition looks like IS the question "which architecture should we use".

In [ ]:
cells = P.winner_cells()
print(f"share of random workloads each arm wins (lambda={P.LAMBDA_HEADLINE:g}, rho={P.RHO_HEADLINE:g}):")
for a, f in cells.items():
    print(f"  {a:11s} {f:6.1%}  {'#' * int(round(f * 50))}")
print()
print("now raise the price of cost:")
for lam in (P.LAMBDA_HEADLINE, 3e-4, 1e-3, 1e-2):
    c2 = P.winner_cells(lam=lam, trials=1200)
    top = max(c2, key=c2.get)
    print(f"  lambda={lam:8.0e} -> {len(c2)} arms hold territory, largest is {top} at {c2[top]:.0%}")
print()
print("A benchmark's cost weighting decides its verdict before any measurement is taken.")

## 4. What no price can buy

The tempting response is to sweep the prices and collect everything that wins somewhere. That does
not work, and the reason is convex geometry rather than retrieval: a linear functional on a finite
point set attains its maximum at an **extreme point of the convex hull**, so an option lying in the
hull's interior is returned by no weights at all.

These are the *unsupported efficient solutions* of multicriteria optimization.

In [ ]:
freq = P.gap_frequency()
print(f"workloads with a Pareto-optimal arm that NO linear price can select: {freq['fraction']:.1%}")
print(f"  which arms go missing: {freq['arms']}")
print()
print("A direct witness — find such a workload and check the missing arm is the argmax for no price:")
P.test_a_gap_arm_is_genuinely_unbuyable()
print("  verified across the full (lambda, rho) grid.")
print()
print("Both standard ways of picking a winner -- aggregate domination and a swept weighted")
print("objective -- fail on the SAME arm, for the same reason: it buys a capability worthless on")
print("most traffic and indispensable on a little of it.")

## 5. And the frontier moves under you anyway

One free parameter is left, and it is the one most often missing from a cost model entirely: the
graph arm holds an index nobody else builds, and charging that build against a deployment of $N$
queries makes its cost per query depend on how big the deployment is.

In [ ]:
print(f"the graph arm's build pays for itself after N* = {P.crossover_n():.2f} queries at K={P.local()['K']}")
print()
print("and that crossover is not a constant either:")
for k in P.K_GRID:
    print(f"  K={k:5d} entities -> N* = {P.crossover_n(k_entities=k):8.1f} queries")
print()
print("The build is quadratic in the entity count while the saving is only linear, so the")
print("deployment must be BIGGER before the index pays for itself as the corpus grows -- the")
print("opposite of the intuition that an index matters more the more there is to index.")

## 6. Every claim, as an assertion

Including the collapse anchors that pin this topic to its predecessor: the table reproduces the
predecessor's columns exactly, an infinite horizon recovers its online-only accounting, and
pricing generation at nothing recovers the routing topic's single-cost model.

In [ ]:
P._run_tests()